# softmax 回归

在机器学习中，我们学习过了 logistic 回归，但是这种回归有一个弊端就是只能处理二分类问题，但是在实际中，我们常常会遇到多分类问题，softmax 回归是 logistic 回归的进一步衍生，用于处理**多分类**问题

再介绍 softmax 回归前，先介绍一下 **独热编码**

## 独热编码
假设我们现在预测的标签有 A、B、C 三个类别，显然为了进行学习，我们需要给这个分类标签进行**编码**，
如果采用简单的数字编码，也就是 `{'A':0,'B':1,'C':2}` 这种编码方式。
如果我们的某个样本真是标签是 A，那么他和 B、C 的距离分别为1,2, 这就使得模型可能会偏向预测到 B，因为这个模型预测 B 的距离比 C 更短，这显然不是一种好的编码方式。

一般的分类问题并不与分类之间的顺序有关。幸运的是，统计学家很早以前就发明了一种分类数据的简单方法：
独热编码 (one-hot encoding)。独热编码是一个向量，它的分量和类别一样多。类别对应的分量设置为 1，其他分量设置为 0。在上面的例子中，标签 $y$ 是一个三维向量:
$$y\in\{(1,0,0),(0,1,0),(0,0,1)\}.$$


## 网络架构图
对于 logistic 回归，我们永远只会得到一个阳性样本的概率即可。
但是对于 softmax 回归，我们需要得到每个样本对应的预测值，所以最终的到的预测值是一个向量
![](../img/softmaxreg.svg)

$$
\begin{aligned}
o_1 &= x_1 w_{11} + x_2 w_{12} + x_3 w_{13} + x_4 w_{14} + b_1,\\
o_2 &= x_1 w_{21} + x_2 w_{22} + x_3 w_{23} + x_4 w_{24} + b_2,\\
o_3 &= x_1 w_{31} + x_2 w_{32} + x_3 w_{33} + x_4 w_{34} + b_3.
\end{aligned}
$$

为了更简洁地表达模型，我们仍然使用线性代数符号。
通过向量形式表达为$\mathbf{o} = \mathbf{W} \mathbf{x} + \mathbf{b}$，
这是一种更适合数学和编写代码的形式。
由此，我们已经将所有权重放到一个$3 \times 4$矩阵中。
对于给定数据样本的特征$\mathbf{x}$，
我们的输出是由权重与输入特征进行矩阵-向量乘法再加上偏置$\mathbf{b}$得到的。

## softmax 的运算
现在我们将优化参数以最大化观测数据的概率。
为了得到预测结果，我们将设置一个阈值，如选择具有最大概率的标签。


我们希望模型的输出$\hat{y}_j$可以视为属于类$j$的概率，
然后选择具有最大输出值的类别$\operatorname*{argmax}_j y_j$作为我们的预测。
例如，如果$\hat{y}_1$、$\hat{y}_2$和$\hat{y}_3$分别为0.1、0.8和0.1，
那么我们预测的类别是2，在我们的例子中代表“鸡”。

然而我们能否将未规范化的预测$o$直接视作我们感兴趣的输出呢？
答案是否定的。
因为将线性层的输出直接视为概率时存在一些问题：
一方面，我们没有限制这些输出数字的总和为1。
另一方面，根据输入的不同，它们可以为负值。



要将输出视为概率，我们必须保证在任何数据上的输出都是非负的且总和为1。
此外，我们需要一个训练的目标函数，来激励模型精准地估计概率。
例如，
在分类器输出0.5的所有样本中，我们希望这些样本是刚好有一半实际上属于预测的类别。
这个属性叫做*校准*（calibration）。

社会科学家邓肯·卢斯于1959年在*选择模型*（choice model）的理论基础上
发明的*softmax函数*正是这样做的：
softmax函数能够将未规范化的预测变换为非负数并且总和为1，同时让模型保持
可导的性质。
为了完成这一目标，我们首先对每个未规范化的预测求幂，这样可以确保输出非负。
为了确保最终输出的概率值总和为1，我们再让每个求幂后的结果除以它们的总和。如下式：

$$\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{o})\quad \text{其中}\quad \hat{y}_j = \frac{\exp(o_j)}{\sum_k \exp(o_k)}$$

这里，对于所有的$j$总有$0 \leq \hat{y}_j \leq 1$。
因此，$\hat{\mathbf{y}}$可以视为一个正确的概率分布。
softmax运算不会改变未规范化的预测$\mathbf{o}$之间的大小次序，只会确定分配给每个类别的概率。
因此，在预测过程中，我们仍然可以用下式来选择最有可能的类别。
$$
\argmax_j \hat{y}_j = \operatorname*{argmax}_j o_j
$$


## 损失函数
### 对数似然
softmax 函数给出了一个向量 $\hat y$，其中 $\hat y_j$ 是样本属于类别 $j$ 的预测概率。
我们希望 $\hat y_j$ 能够尽可能接近真实的标签 $y_j$，其中 $y_j$ 是一个独热向量。
为了衡量预测 $\hat y$ 与真实标签 $y$ 之间的差异，我们可以使用对数似然
$$
\ell(\hat y, y) = - \sum_j y_j \log \hat    
y_j
$$

这个损失函数通常叫做*交叉熵损失*(cross-entropy loss)。

当我们有一个样本属于类别 $j$ 时，$y_j = 1$，其他类别的 $y_k = 0$。
因此，交叉熵损失可以简化为
$$
\ell(\hat y, y) = - \log \hat y_j
$$

## softmax 及其导数
$$
\begin{aligned}
l(\hat y, y) &= - \sum_j y_j \log \hat y_j \\
&= - \sum_j y_j \log \frac{\exp(o_j)}{\sum_k \exp(o_k)} \\
&= - \sum_j y_j \log \exp(o_j) + \sum_j y_j \log \sum_k \exp(o_k) \\
\end{aligned}
$$
由于 $\sum_j y_j = 1$，$\log \sum_k \exp(o_k)$是一个常数我们可以将上式简化为： 
$$
l(\hat y, y) = - \sum_j y_j o_j + \log \sum_k \exp(o_k)
$$


